In [8]:
import fitz  # PyMuPDF
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk

class PDFAnnotator:
    def __init__(self, root):
        self.root = root
        self.canvas = tk.Canvas(root, bg='white')
        self.canvas.pack(fill=tk.BOTH, expand=True)
        self.pdf_document = None
        self.page = None
        self.start_x = None
        self.start_y = None
        self.rect = None
        self.labels = []  # List to store label references
        self.boxes = []

        self.open_button = tk.Button(root, text='Open PDF', command=self.open_pdf)
        self.open_button.pack()

        self.save_button = tk.Button(root, text='Save PDF', command=self.save_pdf)
        self.save_button.pack()

        self.text_entry = tk.Entry(root)
        self.text_entry.pack()

        self.save_entry = tk.Entry(root)
        self.save_entry.pack()
        self.save_entry.insert(0, 'annotated.pdf')  # Default name for the saved PDF

        self.canvas.bind('<Button-1>', self.on_button_press)
        self.canvas.bind('<B1-Motion>', self.on_move_press)
        self.canvas.bind('<ButtonRelease-1>', self.on_button_release)

    def open_pdf(self):
        file_path = filedialog.askopenfilename(filetypes=[("PDF files", "*.pdf")])
        if file_path:
            self.pdf_document = fitz.open(file_path)
            self.page = self.pdf_document.load_page(0)
            self.show_page()

    def save_pdf(self):
        pdf_name = self.save_entry.get()
        if self.pdf_document:
            self.pdf_document.save(pdf_name)
        with open('boxes_coordinates.txt', 'w') as f:
            for box in self.boxes:
                f.write(f"{box}\n")
        print("PDF saved as:", pdf_name)
        print("Boxes coordinates saved in boxes_coordinates.txt")
        self.root.quit()

    def show_page(self):
        pix = self.page.get_pixmap()
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img_tk = ImageTk.PhotoImage(image=img)
        self.canvas.create_image(0, 0, image=img_tk, anchor='nw')
        self.canvas.image = img_tk

    def on_button_press(self, event):
        self.start_x = self.canvas.canvasx(event.x)
        self.start_y = self.canvas.canvasy(event.y)
        self.rect = self.canvas.create_rectangle(self.start_x, self.start_y, self.start_x, self.start_y, outline='red', width=2)

    def on_move_press(self, event):
        cur_x, cur_y = (self.canvas.canvasx(event.x), self.canvas.canvasy(event.y))
        self.canvas.coords(self.rect, self.start_x, self.start_y, cur_x, cur_y)

        # Update or create the label on the left side of the rectangle
        text = self.text_entry.get()
        if self.label is not None:
            self.canvas.delete(self.label)
        self.label = self.canvas.create_text(self.start_x - 10, self.start_y, text=text, anchor='e', fill='red')

    def on_button_release(self, event):
        if self.page:
            end_x = self.canvas.canvasx(event.x)
            end_y = self.canvas.canvasy(event.y)
            if end_x != self.start_x and end_y != self.start_y:
                rect = fitz.Rect(self.start_x, self.start_y, end_x, end_y)
                self.page.add_rect_annot(rect)
                # Add the label as text annotation to the PDF
                text = self.text_entry.get()
                text_rect = fitz.Rect(end_x, self.start_y, end_x + 100, self.start_y + 30)
                self.page.insert_textbox(text_rect, text, fontsize=12, color=(1, 0, 0))
                # Store the coordinates of the marked box
                self.boxes.append((self.start_x, self.start_y, end_x, end_y))
                # Store the label reference
                self.labels.append(self.label)

if __name__ == "__main__":
    root = tk.Tk()
    app = PDFAnnotator(root)
    root.mainloop()


Exception in Tkinter callback
Traceback (most recent call last):
  File "/usr/lib/python3.10/tkinter/__init__.py", line 1921, in __call__
    return self.func(*args)
  File "/tmp/ipykernel_290606/1732120808.py", line 72, in on_move_press
    if self.label is not None:
AttributeError: 'PDFAnnotator' object has no attribute 'label'. Did you mean: 'labels'?
Exception in Tkinter callback
Traceback (most recent call last):
  File "/usr/lib/python3.10/tkinter/__init__.py", line 1921, in __call__
    return self.func(*args)
  File "/tmp/ipykernel_290606/1732120808.py", line 72, in on_move_press
    if self.label is not None:
AttributeError: 'PDFAnnotator' object has no attribute 'label'. Did you mean: 'labels'?
Exception in Tkinter callback
Traceback (most recent call last):
  File "/usr/lib/python3.10/tkinter/__init__.py", line 1921, in __call__
    return self.func(*args)
  File "/tmp/ipykernel_290606/1732120808.py", line 72, in on_move_press
    if self.label is not None:
AttributeError: '

PDF saved as: annotated.pdf
Boxes coordinates saved in boxes_coordinates.txt
